# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


# 🧠 Advanced Agentic AI Pipeline Smart Assistant

## Project Overview
This project implements a single-agent AI pipeline designed to intelligently parse user queries, determine the underlying intent, and conditionally route them to the appropriate tool or a large language model (LLM).

### 🛠️ Key Components:
1. **Secure Authentication:** Uses hidden credential prompts to handle API keys safely.
2. **Intent Routing:** Evaluates queries dynamically to select the execution path.
3. **Tool Integration:** Uses local specialized functions for math and token parsing.
4. **General Inference:** Integrates the Google Gemini 2.5 API for cognitive tasks.
5. **Modern User Interface:** Built using a custom dark neon Cyberpunk Gradio theme for smooth interactions.

In [7]:
# Install required packages smoothly
!pip install -q google-genai gradio

import os
import json
from getpass import getpass
from google import genai
from google.genai import types

# Securely request the user API Key without exposing characters in the terminal
api_key = getpass("Enter your Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Initialize the official Gemini Client platform
try:
    client = genai.Client()
    print("🚀 Gemini Client successfully initialized!")
except Exception as e:
    print(f"❌ Initialization failed. Please verify your credentials: {e}")

Enter your Gemini API Key: ··········
🚀 Gemini Client successfully initialized!


## 🛠️ Specialized Tools Implementation
We define standard granular operations below to execute logic locally without needing external API dependencies for basic constraints.

In [2]:
# 🧮 TOOL 1: Safe Math Calculator
def calculator(expression: str) -> str:
    """Safely evaluates basic numerical calculations."""
    try:
        # Sanitize input: allow only standard numerical characters and operators
        clean_expr = "".join(c for c in expression if c in "0123456789+-*/(). ")
        return str(eval(clean_expr))
    except Exception:
        return "Error in calculation parsing."

# 📝 TOOL 2: Text Keyword Extractor
def extract_keywords(text: str) -> list:
    """Extracts unique relevant keywords longer than 4 characters from raw strings."""
    try:
        words = text.split()
        # Filter filler words and clean structural symbols
        keywords = list(set([w.lower().strip(".,!?;:") for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Main Routing Agent Execution Engine
This block implements the condition-based core logic handler that checks for specific tokens or routes general inputs directly into the Gemini model.

In [3]:
def agent(query: str) -> dict:
    """Core routing engine to processes the query into a standard JSON signature."""
    query_lower = query.lower()

    # Route 1: Math Calculator Pipeline
    if "calculate" in query_lower or "compute" in query_lower:
        expr = query.replace("Calculate", "").replace("calculate", "").replace("compute", "").strip()
        result = calculator(expr)
        return {
            "type": "calculation",
            "result": result
        }

    # Route 2: String Processing / Keywords Pipeline
    elif "keywords" in query_lower or "extract" in query_lower:
        text_content = query.replace("Extract keywords from", "").replace("keywords", "").strip()
        result = extract_keywords(text_content)
        return {
            "type": "keywords",
            "result": result
        }

    # Route 3: Dynamic LLM Generation
    else:
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=query
            )
            return {
                "type": "general",
                "result": response.text.strip()
            }
        except Exception as e:
            return {
                "type": "error",
                "result": f"System error during model invocation: {str(e)}"
            }

## 🎨 Advanced Cyberpunk Chatbot User Interface
Launches a responsive Gradio interface injected with custom CSS variables to provide a high-fidelity experience inside Google Colab notebooks.

In [4]:
import gradio as gr

# Modern Cyberpunk dark UI styling configurations
custom_css = """
body, .gradio-container {
    background: linear-gradient(135deg, #0f172a 0%, #1e1b4b 50%, #311042 100%) !important;
    font-family: 'Segoe UI', system-ui, sans-serif !important;
}
.gradio-container h1 {
    color: #00f2fe !important;
    text-shadow: 0 0 12px rgba(0, 242, 254, 0.4) !important;
    font-weight: 800 !important;
    text-align: center;
}
.gradio-container p {
    color: #94a3b8 !important;
    text-align: center;
}
#chatbot-window {
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    background: rgba(255, 255, 255, 0.02) !important;
    backdrop-filter: blur(16px) !important;
    border-radius: 16px !important;
    box-shadow: 0 12px 40px 0 rgba(0, 0, 0, 0.5) !important;
}
.user {
    background: linear-gradient(90deg, #4f46e5, #06b6d4) !important;
    color: white !important;
    border-radius: 12px 12px 0px 12px !important;
}
.bot {
    background: rgba(255, 255, 255, 0.07) !important;
    color: #f8fafc !important;
    border-radius: 12px 12px 12px 0px !important;
    border: 1px solid rgba(255, 255, 255, 0.05) !important;
}
footer {
    display: none !important;
}
"""

def ui_handler(user_message, history):
    # Execute backend processing logic
    agent_response = agent(user_message)

    route_type = agent_response["type"].upper()
    result_data = agent_response["result"]

    if isinstance(result_data, list):
        formatted_result = ", ".join(f"`{word}`" for word in result_data)
    else:
        formatted_result = str(result_data)

    # Setup conditional dynamic badge styling markers
    badge_color = "#10b981" if route_type == "GENERAL" else ("#f59e0b" if route_type == "CALCULATION" else "#06b6d4")

    html_badge = f"<span style='background-color: {badge_color}; color: black; padding: 3px 8px; border-radius: 5px; font-weight: bold; font-size: 11px;'>🤖 ROUTE: {route_type}</span>"

    return f"{html_badge}\n\n{formatted_result}"

# Render framework block layouts
with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Agentic AI Pipeline Smart Assistant")
    gr.Markdown("An automated multi-intent tool routing engine built with Gemini API.")

    chatbot = gr.Chatbot(elem_id="chatbot-window", height=480)

    gr.ChatInterface(
        fn=ui_handler,
        chatbot=chatbot,
        textbox=gr.Textbox(placeholder="Enter queries (e.g., 'Calculate 105 * 4', 'Extract keywords from...', or basic questions)...", container=False, scale=7),
        examples=[
            "Calculate (1200 / 4) + 50",
            "Extract keywords from Data Science infrastructure optimizes enterprise production workflows",
            "Explain the concept of deep neural networks in two sentences"
        ]
    )

# Run embedded UI application instance
demo.launch(quiet=True, css=custom_css, theme=gr.themes.Soft())

* Running on public URL: https://3c18e486ada944c890.gradio.live
